# Notebook 2 — Benchmark: Impacto do Número de Nós

Mede o tempo de processamento de uma query analítica com 1 e 2 workers Spark ativos.

**Procedimento:**
1. Executar com 2 workers (estado padrão)
2. Parar `datanode-2` no host: `docker stop datanode-2`
3. Executar novamente e comparar
4. Restaurar: `docker start datanode-2`

In [ ]:
import time
import json
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

RESULTS_FILE = '/data/benchmark_results.json'

def load_results():
    if os.path.exists(RESULTS_FILE):
        with open(RESULTS_FILE) as f:
            return json.load(f)
    return []

def save_result(entry):
    results = load_results()
    results.append(entry)
    with open(RESULTS_FILE, 'w') as f:
        json.dump(results, f, indent=2)

print('Utilitários carregados.')

In [ ]:
import subprocess

def contar_workers():
    """Conta workers Spark ativos via API REST."""
    try:
        r = subprocess.run(
            'curl -s http://namenode:8080/json/',
            shell=True, capture_output=True, text=True
        )
        data = json.loads(r.stdout)
        alive = [w for w in data.get('workers', []) if w['state'] == 'ALIVE']
        return len(alive)
    except Exception:
        return -1

workers = contar_workers()
print(f'Workers ativos: {workers}')

In [ ]:
spark = SparkSession.builder \
    .appName(f'Benchmark_{workers}workers') \
    .master('spark://namenode:7077') \
    .config('spark.executor.memory', '1g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'Spark iniciado | Workers: {workers}')

In [ ]:
def executar_query_analitica(df):
    """Query representativa: agregação por categoria e região."""
    return df.groupBy('categoria', 'regiao') \
             .agg(
                 F.count('id').alias('qtd_vendas'),
                 F.sum('total').alias('receita_total'),
                 F.avg('preco_unitario').alias('ticket_medio'),
                 F.max('total').alias('maior_venda')
             ) \
             .orderBy(F.desc('receita_total')) \
             .collect()

resultados = {}
formatos = {
    'CSV':     ('hdfs://namenode:9000/data/csv/vendas.csv', 'csv'),
    'JSON':    ('hdfs://namenode:9000/data/json/vendas.json', 'json'),
    'Parquet': ('hdfs://namenode:9000/data/parquet/vendas.parquet', 'parquet'),
}

for nome, (path, fmt) in formatos.items():
    print(f'\nProcessando {nome}...')
    if fmt == 'csv':
        df = spark.read.option('header', 'true').option('inferSchema', 'true').csv(path)
    elif fmt == 'json':
        df = spark.read.json(path)
    else:
        df = spark.read.parquet(path)

    df.cache()

    # 3 rodadas para estabilizar
    tempos = []
    for rodada in range(3):
        t0 = time.time()
        executar_query_analitica(df)
        elapsed = round(time.time() - t0, 3)
        tempos.append(elapsed)
        print(f'  Rodada {rodada+1}: {elapsed}s')

    df.unpersist()
    media = round(sum(tempos) / len(tempos), 3)
    resultados[nome] = {'tempos': tempos, 'media': media}
    print(f'  Média: {media}s')

    entry = {
        'workers': workers,
        'formato': nome,
        'tempos': tempos,
        'media_segundos': media,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
    }
    save_result(entry)

print('\nBenchmark concluído!')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

all_results = load_results()

# Organiza por (formato, workers)
dados = {}
for r in all_results:
    key = (r['formato'], r['workers'])
    dados[key] = r['media_segundos']

formatos_list = ['CSV', 'JSON', 'Parquet']
workers_list = sorted(set(r['workers'] for r in all_results))
cores = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

x = range(len(formatos_list))
bar_width = 0.8 / max(len(workers_list), 1)

fig, ax = plt.subplots(figsize=(10, 6))

for i, w in enumerate(workers_list):
    valores = [dados.get((f, w), 0) for f in formatos_list]
    offset = (i - len(workers_list) / 2 + 0.5) * bar_width
    bars = ax.bar([xi + offset for xi in x], valores,
                  width=bar_width * 0.9, color=cores[i % len(cores)],
                  label=f'{w} worker(s)')
    for bar, val in zip(bars, valores):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                    f'{val:.1f}s', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Formato do arquivo', fontsize=12)
ax.set_ylabel('Tempo médio (segundos)', fontsize=12)
ax.set_title('Benchmark: Tempo de Processamento por Formato e Número de Workers', fontsize=13)
ax.set_xticks(list(x))
ax.set_xticklabels(formatos_list)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/data/benchmark_chart.png', dpi=150)
print('Gráfico salvo em /data/benchmark_chart.png')
plt.show()

In [ ]:
# Tabela resumo
print('=== RESULTADOS CONSOLIDADOS ===')
print(f'{"Formato":<10} {"Workers":<10} {"Média (s)":<12} {"Timestamp"}')
print('-' * 55)
for r in sorted(all_results, key=lambda x: (x['formato'], x['workers'])):
    print(f"{r['formato']:<10} {r['workers']:<10} {r['media_segundos']:<12.3f} {r['timestamp']}")

spark.stop()